In [2]:
import pandas as pd
from pathlib import Path

INTERIM = Path("../data/interim")
PROCESSED = Path("../data/processed")

dep = pd.read_csv(INTERIM / "dependence_clean.csv")
bdi = pd.read_csv(INTERIM / "depression_clean.csv")
targets = pd.read_csv(INTERIM / "targets_memberB.csv")

print(dep.shape, bdi.shape, targets.shape)

(1220, 26) (1220, 28) (1220, 5)


In [3]:
print((dep["code"] == bdi["code"]).all())
print((dep["code"] == targets["code"]).all())
print(dep["code"].is_unique)

True
True
True


In [4]:
print(targets.head())
print()
targets.info()

   code  depressed_flag  dependence_flag  bdi_total  tmd_total
0     1               0                0          0         28
1     2               0                1          0         54
2     3               0                1          0         46
3     4               0                1          0         41
4     5               0                1          0         40

<class 'pandas.DataFrame'>
RangeIndex: 1220 entries, 0 to 1219
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   code             1220 non-null   int64
 1   depressed_flag   1220 non-null   int64
 2   dependence_flag  1220 non-null   int64
 3   bdi_total        1220 non-null   int64
 4   tmd_total        1220 non-null   int64
dtypes: int64(5)
memory usage: 47.8 KB


In [ ]:
soc = pd.read_csv(INTERIM / "sociodemo_clean.csv") #Opens the sociodemographic data file & names the table "soc"

print(soc.shape) #Prints the dimensions of the soc table
print(soc["code"]. is_unique) #Checks if the student IDs are unique values
print((soc["code"] == targets["code"]).all()) #Checks if the student IDs in soc (member A) match those in targets (member B)
print(soc.isna().sum().sum()) #Prints the total number of missing values in the soc table

(1220, 50)
True
True
0


In [6]:
print(soc.dtypes.value_counts()) #soc.dtypes lists the data types of each column; value_counts() counts the number of columns of each data type
print()
text_cols = soc.select_dtypes(exclude="number").columns.tolist() #Selects columns that are not numeric and converts them to a list
print(text_cols) #Prints the list of text columns in the soc table

int64    33
str      17
Name: count, dtype: int64

['program', 'department', 'sex', 'marital_status', 'ses', 'living_arrangement', 'residence_area', 'subscription_plan', 'network_access', 'areas_use', 'time_first_cellphone', 'type_md', 'hand_dominance', 'usual_phone_location', 'primary_input_finger', 'frequent_touch_gesture', 'age_group']


In [7]:
merged = (
    soc.merge(dep, on="code", validate="one_to_one") #Merges the soc and dep tables on the "code" column, ensuring a one-to-one relationship
    .merge(bdi, on="code", validate="one_to_one") #Merges the result with the bdi table on the "code" column, ensuring a one-to-one relationship
)

print(merged.shape) #Prints the dimensions of the merged table
print(merged["code"].is_unique) #Checks if the student IDs in the merged table are unique values
print(merged.isna().sum().sum()) #Prints the total number of missing values in the merged table

for col in ["depressed_flag", "dependence_flag", "bdi_total", "tmd_total"]:
    print(col, (merged[col].values == targets[col].values).all()) 
 #Checks if the values in the specified columns of the merged table match the corresponding values in the targets table

(1220, 102)
True
0
depressed_flag True
dependence_flag True
bdi_total True
tmd_total True


In [8]:
#value_counts() counts the occurrences of each unique value in the specified column
print(merged["depressed_flag"].value_counts())
print()
print(merged["dependence_flag"].value_counts())
print()

#mean() gives the shares of 1s in the specified columns; rounding to 3 decimal places
print(merged[["depressed_flag", "dependence_flag"]].mean().round(3))

depressed_flag
0    1002
1     218
Name: count, dtype: int64

dependence_flag
0    671
1    549
Name: count, dtype: int64

depressed_flag     0.179
dependence_flag    0.450
dtype: float64


In [9]:
#Dictionary - For each column it says which word becomes which number.
order_maps = {
    "ses": {"Low-low": 0, "Low": 1, "Low-middle": 2, "Middle": 3, "High-middle": 4, "High": 5},
    "time_first_cellphone": {"<6 months": 0, "6-12 months": 1, "1-3 years": 2, "3-6 years": 3, ">6 years": 4},
    "age_group": {"16-17": 0, "18-20": 1, "21-25": 2, "26-35": 3, "36+": 4},
}

#Create a copy of the merged table to apply the encoding without modifying the original data
encoded = merged.copy()

#Apply the encoding for each column based on the order_maps dictionary
#map() replaces the original values with the corresponding numbers from the mapping dictionary
for col, mapping in order_maps.items():
    encoded[col] = encoded[col].map(mapping)

print(encoded[list(order_maps)].head())

#isna().sum() counts the number of blanks. 
#If a word was spelled differently from our lookup book, map would leave a blank
print(encoded[list(order_maps)].isna().sum())

   ses  time_first_cellphone  age_group
0    1                     0          1
1    1                     0          1
2    0                     0          1
3    1                     0          2
4    1                     0          1
ses                     0
time_first_cellphone    0
age_group               0
dtype: int64


In [10]:
#List of columns to be one-hot encoded
unordered_cols = [
    "department", "sex", "marital_status", "living_arrangement",
    "residence_area", "subscription_plan", "network_access",
    "areas_use", "type_md",
]

#pd.get_dummies(...) is the pandas tool that builds the yes/no columns and removes the original word column
#drop_first=True leaves out one answer per column.
#dtype=int makes the new columns 0 and 1, instead of True and False
encoded = pd.get_dummies(encoded, columns=unordered_cols, drop_first=True, dtype=int)


print(encoded.shape)

#Lists any columns that are still words
print(encoded.select_dtypes(exclude="number").columns.tolist())

(1220, 117)
['program', 'hand_dominance', 'usual_phone_location', 'primary_input_finger', 'frequent_touch_gesture', 'tmd_level', 'bdi_level']


In [11]:
from sklearn.model_selection import train_test_split

SEED = 42

strat = encoded["depressed_flag"].astype(str) + "_" + encoded["dependence_flag"].astype(str)

train, test = train_test_split(
    encoded, test_size=0.2, random_state=SEED, stratify=strat
)

print(train.shape, test.shape)
print(train[["depressed_flag", "dependence_flag"]].mean().round(3))
print(test[["depressed_flag", "dependence_flag"]].mean().round(3))

(976, 117) (244, 117)
depressed_flag     0.179
dependence_flag    0.450
dtype: float64
depressed_flag     0.176
dependence_flag    0.451
dtype: float64


In [12]:
from sklearn.preprocessing import StandardScaler

scale_cols = (
    ["semester", "age", "ses", "time_first_cellphone", "age_group",
     "family_expression", "family_activities", "mobile_use_rules",
     "phone_discrimination", "phone_related_arguments"]
    + [f"tmd_q{i}" for i in range(1, 23)]
    + ["tmd_total"]
)

scaler = StandardScaler()
scaler.fit(train[scale_cols])

train_scaled = train.copy()
test_scaled = test.copy()
train_scaled[scale_cols] = scaler.transform(train[scale_cols])
test_scaled[scale_cols] = scaler.transform(test[scale_cols])

print(len(scale_cols), train_scaled.shape, test_scaled.shape)
print(train_scaled[scale_cols[:3]].mean().round(3))
print(test_scaled[scale_cols[:3]].mean().round(3))
print(scaler.mean_[:3].round(2), scaler.scale_[:3].round(2))

33 (976, 117) (244, 117)
semester    0.0
age         0.0
ses        -0.0
dtype: float64
semester   -0.090
age        -0.070
ses        -0.035
dtype: float64
[ 4.93 23.54  2.01] [2.86 6.33 1.13]


In [13]:
PROCESSED.mkdir(parents=True, exist_ok=True)

encoded.to_csv(PROCESSED / "model_ready.csv", index=False)
train.to_csv(PROCESSED / "train.csv", index=False)
test.to_csv(PROCESSED / "test.csv", index=False)
train_scaled.to_csv(PROCESSED / "train_scaled.csv", index=False)
test_scaled.to_csv(PROCESSED / "test_scaled.csv", index=False)

for name in ["model_ready", "train", "test", "train_scaled", "test_scaled"]:
    df = pd.read_csv(PROCESSED / f"{name}.csv")
    print(name, df.shape, df["code"].is_unique, df.isna().sum().sum())

tr = pd.read_csv(PROCESSED / "train.csv")
te = pd.read_csv(PROCESSED / "test.csv")
print("students in both piles:", len(set(tr["code"]) & set(te["code"])))

model_ready (1220, 117) True 0
train (976, 117) True 0
test (244, 117) True 0
train_scaled (976, 117) True 0
test_scaled (244, 117) True 0
students in both piles: 0
